# Who Ruled Formula 1? A 75-Year Data Story

## Imports

In [1]:
import pandas as pd
import sqlite3 
import numpy as np

In [2]:
conn = sqlite3.connect("db/formula1.db")

## 1. The Big Picture

In [24]:
# How has Formula 1 grown over the decades?

query = """ 
WITH decade_stats AS (
        SELECT 
        (year / 10) * 10 AS decade,
        COUNT(DISTINCT ra.raceId) AS total_races,
        COUNT(DISTINCT ra.circuitId) AS unique_circuits,
        COUNT(DISTINCT re.driverId) AS unique_drivers,
        COUNT(DISTINCT re.constructorId) AS unique_constructors
        FROM races ra
        JOIN results re ON ra.raceId = re.raceId
        GROUP BY (year / 10) * 10
)
        SELECT
        decade,
        total_races,
        unique_circuits,
        unique_drivers,
        unique_constructors,
        COALESCE( total_races - LAG(total_races) OVER (ORDER BY decade), 0) AS races_growth
        FROM decade_stats
        ORDER BY decade
        """

pd.read_sql(query, conn)

,decade,total_races,unique_circuits,unique_drivers,unique_constructors,races_growth
0,1950,84,19,332,69,0
1,1960,100,25,219,71,16
2,1970,144,28,173,55,44
3,1980,156,31,115,31,12
4,1990,162,26,105,33,6
5,2000,174,24,71,23,12
6,2010,198,27,66,19,24
7,2020,107,30,36,14,-91


In [28]:
# Which decades had the most races in Formula 1 history?

query = """ 
        SELECT 
        (year / 10) * 10 AS decade,
        RANK() OVER(ORDER BY COUNT(DISTINCT ra.raceId) DESC) as busiest_decade_rank,
        COUNT(DISTINCT ra.raceId) AS total_races
        FROM races ra
        GROUP BY (year / 10) * 10
        ORDER BY decade
        """

pd.read_sql(query, conn)

,decade,busiest_decade_rank,total_races
0,1950,8,84
1,1960,7,100
2,1970,5,144
3,1980,4,156
4,1990,3,162
5,2000,2,174
6,2010,1,198
7,2020,6,107


In [27]:
# Quick summary statistics
query = """
    SELECT
        COUNT(DISTINCT raceId) AS total_races,
        COUNT(DISTINCT circuitId) AS total_circuits,
        MIN(year) AS first_season,
        MAX(year) AS last_season
    FROM races
"""

pd.read_sql(query, conn)

,total_races,total_circuits,first_season,last_season
0,1125,77,1950,2024


## 2. Constructor Dynasties

In [54]:
#How many race entries have the most popular constructors made in F1 history?

query = """ 
with constructor_stats as(
        SELECT 
        c.name,
        count(*) as total_entries
        FROM constructors c
        INNER JOIN results re on re.constructorId=c.constructorId
        GROUP BY c.constructorId
)
        SELECT 
        RANK() OVER(ORDER BY total_entries DESC) as entry_rank,
        name,
        total_entries
        FROM constructor_stats
        LIMIT 10
        """

pd.read_sql(query, conn)

,entry_rank,name,total_entries
0,1,Ferrari,2439
1,2,McLaren,1923
2,3,Williams,1676
3,4,Tyrrell,881
4,5,Team Lotus,871
5,6,Sauber,837
6,7,Red Bull,788
7,8,Renault,787
8,9,Minardi,672
9,10,Brabham,662


In [66]:
# Which constructors won the most races?

query = """ 
WITH constructor_wins AS (
        SELECT 
        (ra.year / 10) * 10 AS decade,
        c.name,
        COUNT(*) AS total_wins
        FROM constructors c
        INNER JOIN results re ON re.constructorId = c.constructorId
        INNER JOIN races ra ON re.raceId = ra.raceId
        WHERE re.position = 1
        GROUP BY c.constructorId, decade
),
ranked AS (
        SELECT 
        RANK() OVER(ORDER BY total_wins DESC) AS win_rank,
        name,
        total_wins
        FROM constructor_wins
)
        SELECT *
        FROM ranked
        WHERE win_rank <= 3
        ORDER BY win_rank
"""

pd.read_sql(query, conn)

,win_rank,name,total_wins
0,1,Mercedes,93
1,2,Ferrari,85
2,3,Williams,61


In [55]:
# Which constructors dominated (in terms of total_wins) each decade of Formula 1?

query = """ 
WITH constructor_wins AS (
        SELECT 
        (ra.year / 10) * 10 AS decade,
        c.name,
        COUNT(*) AS total_wins
        FROM constructors c
        INNER JOIN results re ON re.constructorId = c.constructorId
        INNER JOIN races ra ON re.raceId = ra.raceId
        WHERE re.position = 1
        GROUP BY c.constructorId, decade
),
ranked AS (
        SELECT 
        decade,
        RANK() OVER(PARTITION BY decade ORDER BY total_wins DESC) AS win_rank,
        name,
        total_wins
        FROM constructor_wins
)
        SELECT *
        FROM ranked
        WHERE win_rank <= 3
        ORDER BY decade, win_rank
"""

pd.read_sql(query, conn)

,decade,win_rank,name,total_wins
0,1950,1,Ferrari,30
1,1950,2,Alfa Romeo,11
2,1950,3,Vanwall,10
3,1960,1,Lotus-Climax,22
4,1960,2,Ferrari,13
5,1960,3,BRM,12
6,1970,1,Ferrari,37
7,1970,2,Team Lotus,35
8,1970,3,Tyrrell,21
9,1980,1,McLaren,56


## 3. Greatest Drivers